# BirdCLEF+ 2026 — Perch v2 Baseline

Google Perch v2 の **1536次元 Embedding** を特徴量として抽出し、
**LogisticRegression** で分類するシンプルなベースライン。

## Approach
1. Perch v2 (EfficientNet-B3, 154万録音で事前学習) で全訓練音声の Embedding を抽出
2. sklearn LogisticRegression で多クラス分類器を学習
3. テストサウンドスケープを5秒ウィンドウで推論

## Reference
- LB 0.908 (yashanathaniel, Perch v2 inference notebook)
- Paper: "Perch 2.0: The Bittern Lesson for Bioacoustics" (arXiv, 2025)

## Kaggle Settings
| Setting | Value |
|---------|-------|
| Data | `birdclef-2026` (Competition) |
| Models | `google/bird-vocalization-classifier` → `perch_v2` |
| Accelerator | GPU T4 x2 (recommended) |
| Internet | ON |

In [ ]:
!pip install -q "tensorflow>=2.20" perch-hoplite librosa

In [ ]:
import os
import glob
import pickle
import warnings

import numpy as np
import pandas as pd
import librosa
import tensorflow as tf
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import average_precision_score
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')

print(f'TensorFlow: {tf.__version__}')
print(f'GPU: {tf.config.list_physical_devices("GPU")}')

In [ ]:
# ============================================================
# Path Configuration
# ============================================================
COMP_DATA = None
for p in ['/kaggle/input/birdclef-2026',
          '/kaggle/input/competitions/birdclef-2026']:
    if os.path.exists(p):
        COMP_DATA = p
        break
assert COMP_DATA, 'Competition data not found. Add birdclef-2026 as input.'

# Find Perch v2 SavedModel — search broadly under /kaggle/input
PERCH_MODEL_PATH = None

# 1) glob for saved_model.pb anywhere under /kaggle/input
saved_models = glob.glob('/kaggle/input/**/saved_model.pb', recursive=True)
# Prefer perch_v2 path if multiple found
for sm in saved_models:
    if 'perch' in sm.lower():
        PERCH_MODEL_PATH = os.path.dirname(sm)
        break
if PERCH_MODEL_PATH is None and saved_models:
    PERCH_MODEL_PATH = os.path.dirname(saved_models[0])

# 2) Fallback: try known path patterns
if PERCH_MODEL_PATH is None:
    candidates = [
        '/kaggle/input/bird-vocalization-classifier/tensorflow2/perch_v2/2',
        '/kaggle/input/bird-vocalization-classifier/tensorflow2/perch_v2/1',
        '/kaggle/input/bird-vocalization-classifier/tensorFlow2/perch_v2/2',
        '/kaggle/input/bird-vocalization-classifier/tensorFlow2/perch_v2/1',
        '/kaggle/input/bird-vocalization-classifier/tensorflow2/default/1',
    ]
    for p in candidates:
        if os.path.exists(p):
            PERCH_MODEL_PATH = p
            break

# 3) Debug: show what's actually in /kaggle/input if still not found
if PERCH_MODEL_PATH is None:
    print('=== DEBUG: /kaggle/input contents ===')
    for root, dirs, files in os.walk('/kaggle/input'):
        depth = root.replace('/kaggle/input', '').count(os.sep)
        if depth < 4:  # don't go too deep
            indent = '  ' * depth
            print(f'{indent}{os.path.basename(root)}/')
            for f in files[:5]:
                print(f'{indent}  {f}')
    raise FileNotFoundError(
        'Perch v2 model not found. '
        'Add google/bird-vocalization-classifier (perch_v2) as Model input.'
    )

SAMPLE_RATE = 32000
DURATION = 5
WINDOW_SAMPLES = SAMPLE_RATE * DURATION  # 160,000
OUTPUT_DIR = '/kaggle/working'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f'Competition data: {COMP_DATA}')
print(f'Perch v2 model:   {PERCH_MODEL_PATH}')

## 1. Load Perch v2 Model

In [ ]:
# ============================================================
# Load Perch v2 and define embedding function
# Strategy: try perch-hoplite first, fallback to raw TF
# ============================================================

def load_perch_model(model_path):
    """Load Perch v2 and return an embedding function.
    
    Returns:
        embed_fn: callable that takes waveform (160000,) -> embedding (1536,)
    """
    # --- Approach 1: perch-hoplite (official library) ---
    try:
        from perch_hoplite.zoo import model_configs
        model = model_configs.load_model_by_name(
            'perch_v2', model_path=model_path
        )
        
        def embed_fn(waveform):
            outputs = model.embed(waveform)
            emb = np.array(outputs.embeddings)
            # Spatial embeddings (5,3,1536) -> mean pool to (1536,)
            if emb.ndim > 1:
                emb = emb.reshape(-1, emb.shape[-1]).mean(axis=0)
            return emb.astype(np.float32)
        
        print('Loaded via perch-hoplite')
        return embed_fn
    except Exception as e:
        print(f'perch-hoplite failed: {e}')
    
    # --- Approach 2: Raw TF SavedModel ---
    model = tf.saved_model.load(model_path)
    
    # Inspect signatures
    if hasattr(model, 'signatures') and 'serving_default' in model.signatures:
        infer_fn = model.signatures['serving_default']
        output_keys = list(infer_fn.structured_outputs.keys())
        print(f'TF SavedModel output keys: {output_keys}')
        
        # Find the embedding output key
        embed_key = None
        for k in output_keys:
            if 'embed' in k.lower():
                embed_key = k
                break
        if embed_key is None:
            embed_key = output_keys[0]
        
        def embed_fn(waveform):
            wf = tf.constant(waveform[np.newaxis], dtype=tf.float32)
            result = infer_fn(wf)
            emb = result[embed_key].numpy()
            if emb.ndim > 2:
                emb = emb.reshape(-1, emb.shape[-1]).mean(axis=0)
            elif emb.ndim == 2:
                emb = emb[0]
            return emb.astype(np.float32)
        
        print(f'Loaded via TF SavedModel (output key: {embed_key})')
        return embed_fn
    
    # --- Approach 3: Direct __call__ ---
    def embed_fn(waveform):
        wf = tf.constant(waveform[np.newaxis], dtype=tf.float32)
        result = model(wf)
        if isinstance(result, dict):
            for k, v in result.items():
                if 'embed' in k.lower():
                    emb = v.numpy()
                    break
            else:
                emb = list(result.values())[0].numpy()
        else:
            emb = result.numpy()
        if emb.ndim > 2:
            emb = emb.reshape(-1, emb.shape[-1]).mean(axis=0)
        elif emb.ndim == 2:
            emb = emb[0]
        return emb.astype(np.float32)
    
    print('Loaded via TF direct call')
    return embed_fn


embed_fn = load_perch_model(PERCH_MODEL_PATH)

In [ ]:
# Quick test: embed a silent 5-second clip
test_wf = np.zeros(WINDOW_SAMPLES, dtype=np.float32)
test_emb = embed_fn(test_wf)
print(f'Embedding shape: {test_emb.shape}')   # expect (1536,)
print(f'Embedding dtype: {test_emb.dtype}')
print(f'Embedding range: [{test_emb.min():.4f}, {test_emb.max():.4f}]')
assert test_emb.shape == (1536,), f'Unexpected shape: {test_emb.shape}'
print('Embedding extraction OK')

## 2. Extract Training Embeddings

In [ ]:
# ============================================================
# Load metadata
# ============================================================
train_df = pd.read_csv(f'{COMP_DATA}/train.csv')
sample_sub = pd.read_csv(f'{COMP_DATA}/sample_submission.csv')
species_cols = [c for c in sample_sub.columns if c != 'row_id']

print(f'Training samples: {len(train_df)}')
print(f'Submission species: {len(species_cols)}')
print(f'Training classes:  {train_df["primary_label"].nunique()}')

In [ ]:
# ============================================================
# Extract 1536-dim embedding for every training sample
# Takes ~30-60 min on GPU T4 (35,549 files)
# ============================================================

def load_waveform(audio_path, sr=SAMPLE_RATE, duration=DURATION):
    """Load audio, pad/crop to exactly `duration` seconds."""
    try:
        waveform, _ = librosa.load(audio_path, sr=sr, duration=duration)
    except Exception:
        return np.zeros(sr * duration, dtype=np.float32)
    target_len = sr * duration
    if len(waveform) < target_len:
        waveform = np.pad(waveform, (0, target_len - len(waveform)))
    return waveform[:target_len].astype(np.float32)


embeddings = np.zeros((len(train_df), 1536), dtype=np.float32)

for i, row in tqdm(train_df.iterrows(), total=len(train_df),
                    desc='Extracting embeddings'):
    path = os.path.join(COMP_DATA, 'train_audio', row['filename'])
    wf = load_waveform(path)
    embeddings[i] = embed_fn(wf)

print(f'Embeddings shape: {embeddings.shape}')  # (35549, 1536)
print(f'NaN count: {np.isnan(embeddings).sum()}')

## 3. Train Classifier

In [ ]:
# ============================================================
# Label Encoding
# ============================================================
le = LabelEncoder()
y_train = le.fit_transform(train_df['primary_label'].values)
n_classes = len(le.classes_)
print(f'Classes: {n_classes}')

# ============================================================
# 5-Fold CV with LogisticRegression
# ============================================================
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = []

for fold, (tr_idx, va_idx) in enumerate(skf.split(embeddings, y_train)):
    clf = LogisticRegression(
        max_iter=1000, C=0.1, solver='lbfgs',
        multi_class='multinomial', n_jobs=-1
    )
    clf.fit(embeddings[tr_idx], y_train[tr_idx])
    
    # predict_proba may have fewer columns if rare classes are missing
    # from this fold's training set. Build a full (N, n_classes) array.
    raw_probs = clf.predict_proba(embeddings[va_idx])
    val_probs = np.zeros((len(va_idx), n_classes), dtype=np.float32)
    for i, c in enumerate(clf.classes_):
        val_probs[:, c] = raw_probs[:, i]
    
    # Compute class-mean Average Precision (cmAP)
    aps = []
    for c in range(n_classes):
        y_true_c = (y_train[va_idx] == c).astype(int)
        if y_true_c.sum() > 0:
            ap = average_precision_score(y_true_c, val_probs[:, c])
            aps.append(ap)
    cmap = np.mean(aps)
    cv_scores.append(cmap)
    print(f'  Fold {fold}: cmAP = {cmap:.4f}')

print(f'\nCV Mean cmAP: {np.mean(cv_scores):.4f} (+/- {np.std(cv_scores):.4f})')

# ============================================================
# Train final model on all data
# ============================================================
clf_final = LogisticRegression(
    max_iter=1000, C=0.1, solver='lbfgs',
    multi_class='multinomial', n_jobs=-1
)
clf_final.fit(embeddings, y_train)
print(f'Final model trained on {len(embeddings)} samples')

# Save for inference notebook
with open(f'{OUTPUT_DIR}/perch_v2_classifier.pkl', 'wb') as f:
    pickle.dump({'clf': clf_final, 'le': le}, f)
print(f'Saved to {OUTPUT_DIR}/perch_v2_classifier.pkl')

## 4. Inference on Test Soundscapes

In [ ]:
# ============================================================
# Build species mapping (submission columns -> classifier index)
# ============================================================
species_to_idx = {}
for sp in species_cols:
    try:
        species_to_idx[sp] = int(le.transform([sp])[0])
    except ValueError:
        pass  # zero-shot species (not in training data)

print(f'Mapped species: {len(species_to_idx)} / {len(species_cols)}')
print(f'Zero-shot species: {len(species_cols) - len(species_to_idx)}')

# ============================================================
# Sliding window inference on test soundscapes
# ============================================================
test_dir = f'{COMP_DATA}/test_soundscapes'
test_files = sorted(f for f in os.listdir(test_dir) if f.endswith('.ogg'))
print(f'Test files: {len(test_files)}')

rows = []
for fname in tqdm(test_files, desc='Inference'):
    path = os.path.join(test_dir, fname)
    stem = os.path.splitext(fname)[0]
    
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        full_waveform, _ = librosa.load(path, sr=SAMPLE_RATE, mono=True)
    
    total_samples = len(full_waveform)
    if total_samples == 0:
        continue
    
    # Process each 5-second window
    for start in range(0, total_samples, WINDOW_SAMPLES):
        window = full_waveform[start:start + WINDOW_SAMPLES]
        if len(window) < WINDOW_SAMPLES:
            window = np.pad(window, (0, WINDOW_SAMPLES - len(window)))
        window = window.astype(np.float32)
        
        bucket_sec = (start // WINDOW_SAMPLES) * DURATION
        row_id = f'{stem}_{bucket_sec}'
        
        # Extract embedding and predict
        emb = embed_fn(window)
        probs = clf_final.predict_proba(emb.reshape(1, -1))[0]
        
        row = {'row_id': row_id}
        for sp in species_cols:
            if sp in species_to_idx:
                row[sp] = float(probs[species_to_idx[sp]])
            else:
                row[sp] = 0.0  # zero-shot species
        rows.append(row)

print(f'Total predictions: {len(rows)}')

In [ ]:
# ============================================================
# Build submission
# ============================================================
submission = pd.DataFrame(rows, columns=['row_id'] + species_cols)

# Align with sample submission (fill missing row_ids with 0)
submission = (
    submission
    .set_index('row_id')
    .reindex(sample_sub['row_id'], fill_value=0.0)
    .reset_index()
)

out_path = f'{OUTPUT_DIR}/submission.csv'
submission.to_csv(out_path, index=False)

print(f'Submission saved: {out_path}')
print(f'  Rows: {len(submission)}')
print(f'  Columns: {len(submission.columns)}')
submission.head(3)

In [ ]:
# ============================================================
# Verify submission
# ============================================================
assert len(submission) == len(sample_sub), \
    f'Row count mismatch: {len(submission)} vs {len(sample_sub)}'
assert list(submission.columns) == list(sample_sub.columns), \
    'Column mismatch'
assert submission.isnull().sum().sum() == 0, \
    'NaN values found'

score_vals = submission.drop('row_id', axis=1).values
print('Submission verification OK')
print(f'  Mean prediction:  {score_vals.mean():.6f}')
print(f'  Max prediction:   {score_vals.max():.4f}')
print(f'  Zero rate:        {(score_vals == 0.0).mean():.2%}')
print(f'  Non-zero species: {(score_vals.max(axis=0) > 0).sum()} / {len(species_cols)}')

In [ ]:
# ============================================================
# Convert Perch v2 SavedModel to TFLite
# TFLite strips XlaCallModuleOp, so it works on any TF version
# ============================================================
import tensorflow as tf

print(f'Converting SavedModel: {PERCH_MODEL_PATH}')

converter = tf.lite.TFLiteConverter.from_saved_model(PERCH_MODEL_PATH)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.target_spec.supported_ops = [
    tf.lite.OpsSet.TFLITE_BUILTINS,
    tf.lite.OpsSet.SELECT_TF_OPS,
]

tflite_model = converter.convert()

tflite_path = f'{OUTPUT_DIR}/perch_v2.tflite'
with open(tflite_path, 'wb') as f:
    f.write(tflite_model)

print(f'TFLite saved: {tflite_path} ({len(tflite_model) / 1024 / 1024:.1f} MB)')

# Verify TFLite works
interpreter = tf.lite.Interpreter(model_path=tflite_path)
interpreter.allocate_tensors()
input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()
print(f'Input:  {input_details[0]["shape"]} {input_details[0]["dtype"]}')
for od in output_details:
    print(f'Output: {od["name"]} {od["shape"]}')

# Test embedding extraction via TFLite
test_wf = np.zeros(WINDOW_SAMPLES, dtype=np.float32)
interpreter.set_tensor(input_details[0]['index'], test_wf.reshape(1, -1))
interpreter.invoke()

# Find embedding output (1536-dim)
for od in output_details:
    result = interpreter.get_tensor(od['index'])
    if result.shape[-1] == 1536:
        print(f'Embedding output key: {od["name"]}, shape: {result.shape}')
        break

print('TFLite conversion & verification OK')

## 5. Convert Perch to TFLite (for offline submission)